# B_S3.4 - Metric Selection Framed by Permutation p-values and Alpha

Goal: choose a small metric set for relationship detection using the standard permutation-test decision rule:

```text
p <= alpha  -> detectable relationship
p > alpha   -> no detectable relationship
```

**Important**: this notebook restricts to the **MINE-covered subset** (~9K cases) so that all 44 metrics are evaluated on exactly the same cases. This avoids the coverage bias where MINE/LOWESS metrics (computed on 9K) would be unfairly penalized against Phase 1-3 metrics (computed on 17K).

Core idea:

1. For each metric or metric combination, define a test statistic `T`.
2. Use permutations of `y` to obtain the null distribution of `T` under independence.
3. Compare `T_obs` with that null distribution to get a permutation p-value.
4. Choose `alpha`, usually 0.05.
5. Classify cases by `p <= alpha`.

Important terminology:

- `alpha`: the p-value cutoff used for declaring a detectable relationship.
- `validation_FPR`: inherited output name; read it as the fraction of validation `true_null` cases with `p <= alpha`.
- `validation_overall_power`: fraction of validation signal cases with `p <= alpha`.

So FPR is not a separate method. It is just what happens when the p-value rule is applied to known no-relationship cases.

In [ ]:
from pathlib import Path
from itertools import combinations
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

S1_DIR = Path('output/S1')
S3_DIR = Path('output/S3')
OUT_DIR = Path('output/S3.4')
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 20260623
ALPHA = 0.05
# Set False to use all data for both fitting and evaluation (no train/validation split).
USE_SPLIT = True
TRAIN_FRACTION = 0.5

# Exact exhaustive subset search is only feasible for small k.
MAX_EXACT_K = 3
MAX_EXACT_COMBOS = 75_000
MAX_SPARSE_K = 8
BEAM_WIDTH = 60
NEAR_FULL_TOL = 0.02
VALIDATION_FPR_LIMIT = 0.08

# ── Metric filter toggles (shared with S3.0, S3.2, S3.3) ──
EXCLUDE_REVERSED = True   # 8 bw metrics: z↑ = no relationship, wrong for max-Z
EXCLUDE_WEAK     = True   # 9 metrics with signal/null separation < 0.5
EXCLUDE_RAW      = True   # 12 scale-dependent unbounded metrics
MERGE_DCOR_DCOV  = True   # remove dcov (mathematically redundant with dcor)

# Category weights for weighted macro power
CATEGORY_WEIGHTS = {'mean_only': 2, 'variance_only': 1, 'mean+variance': 2}

## Decision Rule

For a selected metric combination `S`, the conceptual test is:

```text
T_obs = max(Z_m for m in S)
T_null,b = max(Z_null,m,b for m in S)
p = (1 + number of permutations where T_null,b >= T_obs) / (B + 1)
```

Then:

```text
p <= alpha  -> detectable relationship
p > alpha   -> no detectable relationship
```

The code often uses the equivalent threshold form:

```text
threshold = 95th percentile of T_null when alpha = 0.05
T_obs > threshold <=> p <= alpha
```

This threshold form makes it easier to compare many candidate metric combinations efficiently.


## 1. Load B_S3 Results and Metadata

Load the full B_S3 output, then **filter to MINE-covered cases only** so all metrics are evaluated on exactly the same set of cases.

In [ ]:
df = pd.read_parquet(S3_DIR / 'permutation_all.parquet')
Z_COLS = sorted([c for c in df.columns if c.startswith('z_')])
P_COLS = sorted([c for c in df.columns if c.startswith('p_') and c != 'p_value'])

cases_main = pd.read_csv(S1_DIR / 'cases.csv', low_memory=False)
cases_null = pd.read_csv(S1_DIR / 'null_expanded_cases.csv', low_memory=False)
cases_null['case_id'] = cases_null['case_id'] + cases_main['case_id'].max()

meta_cols = ['case_id', 'family_id', 'snr', 'spread_pattern', 'x_distribution']
cases_all = pd.concat([cases_main[meta_cols], cases_null[meta_cols]], ignore_index=True)
df = df.merge(cases_all, on='case_id', how='left')

is_null_family = df['family_id'] == 'Null'
is_constant_spread = df['spread_pattern'] == 'constant'

df['category'] = 'mean+variance'
df.loc[is_null_family & is_constant_spread, 'category'] = 'true_null'
df.loc[~is_null_family & is_constant_spread, 'category'] = 'mean_only'
df.loc[is_null_family & ~is_constant_spread, 'category'] = 'variance_only'

print(f'B_S3 output (before filter): {len(df):,} cases x {len(Z_COLS)} Z-score metrics')

# ── Filter to MINE-covered cases only ──
# Phase 4 (MINE) and Phase 5 (LOWESS) ran on a ~9K subset.
# Keep only cases where all MINE metrics are non-null so every metric
# is evaluated on exactly the same set — no coverage bias.
mine_z_cols = [c for c in Z_COLS if c in ('z_mic', 'z_mas', 'z_mev', 'z_mcn')]
mine_coverage_mask = df[mine_z_cols].notna().all(axis=1)
n_before = len(df)
df = df[mine_coverage_mask].reset_index(drop=True)
print(f'Filtered to MINE-covered cases: {len(df):,} / {n_before:,}')

# Verify all Z columns are now fully covered
n_partial = sum(1 for zc in Z_COLS if df[zc].isna().any())
assert n_partial == 0, f'{n_partial} Z-columns still have NaN after MINE filter!'
print(f'All {len(Z_COLS)} Z-score metrics have full coverage on {len(df):,} cases')

print(f'\nCategories:')
print(df['category'].value_counts().to_string())
print(f'\nSource: {df["source"].value_counts().to_dict()}')

## 2. Metric Display Names and Groups


In [ ]:
def metric_display(col):
    name = col.replace('z_', '')
    replacements = {
        'abs_pearson_r': '|pearson|',
        'abs_spearman_rho': '|spearman|',
        'abs_covariance': '|covariance|',
        'mic_minus_r2': 'MIC−r²',
    }
    return replacements.get(name, name)

METRIC_DISPLAY = {c: metric_display(c) for c in Z_COLS}

METRIC_GROUPS = {}
for c in Z_COLS:
    name = c.replace('z_', '')
    if name in ('pearson', 'spearman', 'abs_pearson_r', 'abs_spearman_rho', 'abs_covariance'):
        METRIC_GROUPS[c] = 'correlation'
    elif 'dcor' in name or 'dcov' in name:
        METRIC_GROUPS[c] = 'distance'
    elif 'ep_' in name or 'pf_' in name or 'seg_' in name:
        METRIC_GROUPS[c] = 'slope'
    elif 'bin_' in name or name == 'eta2':
        METRIC_GROUPS[c] = 'bin'
    elif 'dist_' in name:
        METRIC_GROUPS[c] = 'distribution'
    elif name in ('mic', 'mas', 'mev', 'mcn', 'mic_minus_r2'):
        METRIC_GROUPS[c] = 'mine'
    elif name == 'lowess_r2':
        METRIC_GROUPS[c] = 'nonlinear'
    else:
        METRIC_GROUPS[c] = 'other'

GROUP_COLORS = {
    'correlation': '#e41a1c',
    'distance': '#377eb8',
    'slope': '#4daf4a',
    'bin': '#ff7f00',
    'distribution': '#984ea3',
    'mine': '#a65628',
    'nonlinear': '#f781bf',
    'other': '#999999',
}

print('Metric groups:')
for group in sorted(set(METRIC_GROUPS.values())):
    members = [c for c in Z_COLS if METRIC_GROUPS[c] == group]
    print(f'  {group:12s}: {len(members):2d} metrics')

## 3. Train/Validation Split (Optional)

When `USE_SPLIT=True`, the data is split 50/50 stratified by `category + x_distribution`. All selection uses train only, evaluation uses validation.

When `USE_SPLIT=False`, all data is used for both fitting and evaluation (no held-out set).

In [ ]:
if USE_SPLIT:
    rng = np.random.default_rng(RANDOM_SEED)
    df['split'] = 'validation'

    for _, idx in df.groupby(['category', 'x_distribution'], dropna=False).groups.items():
        idx = np.array(list(idx)).copy()
        rng.shuffle(idx)
        n_train = int(round(len(idx) * TRAIN_FRACTION))
        if len(idx) > 1:
            n_train = min(max(n_train, 1), len(idx) - 1)
        df.loc[idx[:n_train], 'split'] = 'train'

    train = df[df['split'] == 'train'].copy()
    validation = df[df['split'] == 'validation'].copy()

    split_counts = df.groupby(['split', 'category']).size().unstack(fill_value=0)
    print(split_counts)
    print('\nFraction train by category:')
    print((split_counts.loc['train'] / split_counts.sum(axis=0)).to_string(float_format='{:.3f}'.format))
else:
    df['split'] = 'all'
    train = df.copy()
    validation = df.copy()
    print(f'USE_SPLIT=False: using all {len(df):,} cases for both fitting and evaluation')
    print(df['category'].value_counts().to_string())

## 4. Evaluation Helpers

Every combination uses the same statistic:

```text
T_combo = max(Z_metric for metric in combo)
```

The threshold is fit from train true_null only. Since all cases have full metric coverage (MINE filter applied in §1), there are no missing values to handle.

In [ ]:
def fit_threshold(data, metrics, alpha=ALPHA):
    null_data = data[data['category'] == 'true_null']
    T_null = null_data[metrics].max(axis=1).dropna()
    if len(T_null) == 0:
        return np.nan
    return float(np.nanpercentile(T_null, 100 * (1 - alpha)))


def evaluate_fixed_threshold(data, metrics, threshold):
    T = data[metrics].max(axis=1)
    detected = (T > threshold).fillna(False)

    out = {
        'n_metrics': len(metrics),
        'threshold': threshold,
        'n_total': len(data),
    }
    for cat in ['true_null', 'mean_only', 'variance_only', 'mean+variance']:
        mask = data['category'] == cat
        out[f'n_{cat}'] = int(mask.sum())
        out[f'rate_{cat}'] = float(detected[mask].mean()) if mask.sum() else np.nan

    signal_mask = data['category'] != 'true_null'
    out['overall_power'] = float(detected[signal_mask].mean()) if signal_mask.sum() else np.nan
    out['detected'] = detected
    out['T'] = T
    return out


def fit_and_evaluate(train_data, eval_data, metrics, alpha=ALPHA):
    threshold = fit_threshold(train_data, metrics, alpha=alpha)
    result = evaluate_fixed_threshold(eval_data, metrics, threshold)
    result['metrics'] = tuple(metrics)
    return result


def result_row(result, prefix):
    mo = result['rate_mean_only']
    vo = result['rate_variance_only']
    mv = result['rate_mean+variance']
    _w = CATEGORY_WEIGHTS
    _pairs = [(mo, _w['mean_only']), (vo, _w['variance_only']), (mv, _w['mean+variance'])]
    _valid = [(r, wt) for r, wt in _pairs if r is not None and not np.isnan(r)]
    macro = float(sum(r*wt for r,wt in _valid) / sum(wt for _,wt in _valid)) if _valid else np.nan
    return {
        f'{prefix}_overall_power': result['overall_power'],
        f'{prefix}_macro_power': macro,
        f'{prefix}_FPR': result['rate_true_null'],
        f'{prefix}_MO': result['rate_mean_only'],
        f'{prefix}_VO': result['rate_variance_only'],
        f'{prefix}_MV': result['rate_mean+variance'],
    }


def evaluate_combo_train_validation(metrics, label=None, kind=None):
    metrics = list(metrics)
    train_result = fit_and_evaluate(train, train, metrics, alpha=ALPHA)
    val_result = evaluate_fixed_threshold(validation, metrics, train_result['threshold'])
    row = {
        'label': label or ', '.join(METRIC_DISPLAY[m] for m in metrics),
        'kind': kind or 'combo',
        'k': len(metrics),
        'metrics': ','.join(metrics),
        'display_metrics': ', '.join(METRIC_DISPLAY[m] for m in metrics),
        'threshold_from_train': train_result['threshold'],
        'has_mine': any(METRIC_GROUPS[m] == 'mine' for m in metrics),
    }
    row.update(result_row(train_result, 'train'))
    row.update(result_row(val_result, 'validation'))
    return row, train_result, val_result


def print_eval_table(table, max_rows=30):
    cols = ['label', 'k', 'train_overall_power', 'train_FPR',
            'validation_overall_power', 'validation_macro_power', 'validation_FPR',
            'validation_MO', 'validation_VO', 'validation_MV']
    available = [c for c in cols if c in table.columns]
    print(table[available].head(max_rows).to_string(index=False, float_format='{:.3f}'.format))

## 5. Train-Only Calibration Screen

This screen removes degenerate or badly behaved metrics before sparse selection. It uses train true_null only.


In [ ]:
train_null = train[train['category'] == 'true_null']
calib_rows = []
for zc in Z_COLS:
    z_null = train_null[zc].dropna()
    pc = zc.replace('z_', 'p_')
    p_null = train_null[pc].dropna() if pc in train_null.columns else pd.Series(dtype=float)
    empirical_fpr = float((p_null <= ALPHA).mean()) if len(p_null) else np.nan
    calib_rows.append({
        'metric': zc,
        'display': METRIC_DISPLAY[zc],
        'group': METRIC_GROUPS[zc],
        'n_valid_train_null': int(len(z_null)),
        'mean_train_null': float(z_null.mean()) if len(z_null) else np.nan,
        'std_train_null': float(z_null.std()) if len(z_null) > 1 else np.nan,
        'q95_train_null': float(np.nanpercentile(z_null, 95)) if len(z_null) else np.nan,
        'empirical_p_fpr_train_null': empirical_fpr,
    })

calib = pd.DataFrame(calib_rows)
calib['flag_bias'] = calib['mean_train_null'].abs() > 1.0
calib['flag_degen'] = calib['std_train_null'] < 0.3
calib['flag_unstable'] = calib['std_train_null'] > 3.0
calib['flag_high_p_fpr'] = calib['empirical_p_fpr_train_null'] > 0.15
calib['flagged'] = calib[['flag_bias', 'flag_degen', 'flag_unstable', 'flag_high_p_fpr']].any(axis=1)

SAFE_METRICS = calib.loc[~calib['flagged'], 'metric'].tolist()

# ── Apply metric filter toggles ──
_reversed = {c for c in Z_COLS if '_bin_bw_' in c}
_weak = {c for c in Z_COLS
         if any(c.endswith(s) for s in ['_ep_early', '_ep_mid', '_ep_late'])
         or c in ('z_raw_seg_strength', 'z_std_seg_strength', 'z_mcn')}
_raw = {c for c in Z_COLS
        if c.startswith('z_abs_raw_') or c == 'z_raw_seg_strength'
        or c == 'z_abs_covariance' or '_bin_amp' in c}

exclude = set()
if EXCLUDE_REVERSED: exclude |= _reversed
if EXCLUDE_WEAK:     exclude |= _weak
if EXCLUDE_RAW:      exclude |= _raw
if MERGE_DCOR_DCOV and 'z_dcov' in Z_COLS:
    exclude.add('z_dcov')

SAFE_METRICS = [c for c in SAFE_METRICS if c not in exclude]

n_rev = len(_reversed & exclude)
n_weak = len(_weak & exclude)
n_raw = len(_raw & exclude)
print(f'Filter toggles: REVERSED={EXCLUDE_REVERSED} ({n_rev}), '
      f'WEAK={EXCLUDE_WEAK} ({n_weak}), RAW={EXCLUDE_RAW} ({n_raw}), '
      f'DCOV={MERGE_DCOR_DCOV}')
print(f'Safe metrics: {len(SAFE_METRICS)} / {len(Z_COLS)}')
print(f'(All metrics have full coverage after MINE filter — no partial-coverage issue)')

if calib['flagged'].any():
    print('\nFlagged metrics (calibration screen):')
    print(calib.loc[calib['flagged'], ['display', 'group', 'mean_train_null', 'std_train_null', 'empirical_p_fpr_train_null']]
          .sort_values('empirical_p_fpr_train_null', ascending=False)
          .to_string(index=False, float_format='{:.3f}'.format))

calib.to_csv(OUT_DIR / 'train_null_calibration.csv', index=False, float_format='%.6f')

## 6. Full-Combo References

Reference points showing what happens when all metrics are combined. Note: adding more metrics raises the max-Z threshold (multiple-comparison penalty), so sparse subsets can outperform these references.

In [ ]:
benchmark_specs = [
    ('all_metrics', Z_COLS),
    ('all_safe', SAFE_METRICS),
]

benchmark_rows = []
for label, metrics in benchmark_specs:
    if len(metrics) == 0:
        continue
    row, _, _ = evaluate_combo_train_validation(metrics, label=label, kind='benchmark')
    benchmark_rows.append(row)

benchmarks = pd.DataFrame(benchmark_rows).sort_values('validation_overall_power', ascending=False)
benchmarks.to_csv(OUT_DIR / 'full_benchmarks.csv', index=False, float_format='%.6f')
print_eval_table(benchmarks)

FULL_SAFE_POWER = float(benchmarks.loc[benchmarks['label'] == 'all_safe', 'validation_overall_power'].iloc[0])
FULL_SAFE_MACRO_POWER = float(benchmarks.loc[benchmarks['label'] == 'all_safe', 'validation_macro_power'].iloc[0])
print(f'\nFull-combo reference: overall={FULL_SAFE_POWER:.3f}, macro={FULL_SAFE_MACRO_POWER:.3f}')

## 7. Individual Metrics: Best k=1

This answers: which metric alone has the highest held-out performance?


In [ ]:
individual_rows = []
for metric in SAFE_METRICS:
    row, _, _ = evaluate_combo_train_validation([metric], label=METRIC_DISPLAY[metric], kind='individual')
    row['group'] = METRIC_GROUPS[metric]
    individual_rows.append(row)

individual = pd.DataFrame(individual_rows).sort_values('train_macro_power', ascending=False)
individual.to_csv(OUT_DIR / 'individual_metric_train_validation.csv', index=False, float_format='%.6f')

print('Top individual metrics by train power; validation is held out:')
print(individual[['label', 'group', 'train_overall_power', 'train_FPR',
                  'validation_overall_power', 'validation_FPR',
                  'validation_MO', 'validation_VO', 'validation_MV']]
      .head(25).to_string(index=False, float_format='{:.3f}'.format))


## 8. Group-Only and Leave-One-Group-Out Ablation

Group-only asks: how much can this metric family do by itself?

Leave-one-group-out asks: how much does performance drop when this family is removed from the full safe set?


In [ ]:
group_rows = []
groups = sorted(set(METRIC_GROUPS[m] for m in SAFE_METRICS))

for group in groups:
    group_metrics = [m for m in SAFE_METRICS if METRIC_GROUPS[m] == group]
    if group_metrics:
        row, _, _ = evaluate_combo_train_validation(group_metrics, label=f'{group} only', kind='group_only')
        row['group'] = group
        group_rows.append(row)

for group in groups:
    remaining = [m for m in SAFE_METRICS if METRIC_GROUPS[m] != group]
    if remaining:
        row, _, _ = evaluate_combo_train_validation(remaining, label=f'all_safe - {group}', kind='leave_one_group_out')
        row['group_removed'] = group
        row['validation_power_drop_vs_full_safe'] = FULL_SAFE_POWER - row['validation_overall_power']
        group_rows.append(row)

group_ablation = pd.DataFrame(group_rows)
group_ablation.to_csv(OUT_DIR / 'group_ablation_train_validation.csv', index=False, float_format='%.6f')

print('Group-only performance:')
print_eval_table(group_ablation[group_ablation['kind'] == 'group_only'].sort_values('validation_overall_power', ascending=False))

print('\nLeave-one-group-out performance:')
leave_group = group_ablation[group_ablation['kind'] == 'leave_one_group_out'].sort_values('validation_power_drop_vs_full_safe', ascending=False)
cols = ['label', 'k', 'validation_overall_power', 'validation_FPR', 'validation_power_drop_vs_full_safe', 'validation_MO', 'validation_VO', 'validation_MV']
print(leave_group[cols].to_string(index=False, float_format='{:.3f}'.format))


## 9. Leave-One-Metric-Out Ablation

This asks: from the full safe set, which single metric removal causes the biggest validation power drop?

A large drop means the metric has unique contribution not fully covered by the rest.


In [ ]:
metric_ablation_rows = []
for metric in SAFE_METRICS:
    remaining = [m for m in SAFE_METRICS if m != metric]
    if not remaining:
        continue
    row, _, _ = evaluate_combo_train_validation(remaining, label=f'all_safe - {METRIC_DISPLAY[metric]}', kind='leave_one_metric_out')
    row['metric_removed'] = metric
    row['display_removed'] = METRIC_DISPLAY[metric]
    row['group_removed'] = METRIC_GROUPS[metric]
    row['validation_power_drop_vs_full_safe'] = FULL_SAFE_POWER - row['validation_overall_power']
    metric_ablation_rows.append(row)

metric_ablation = pd.DataFrame(metric_ablation_rows).sort_values('validation_power_drop_vs_full_safe', ascending=False)
metric_ablation.to_csv(OUT_DIR / 'metric_ablation_train_validation.csv', index=False, float_format='%.6f')

print('Top leave-one-metric-out drops:')
print(metric_ablation[['display_removed', 'group_removed', 'validation_power_drop_vs_full_safe',
                       'validation_overall_power', 'validation_FPR', 'validation_MO', 'validation_VO', 'validation_MV']]
      .head(25).to_string(index=False, float_format='{:.4f}'.format))


## 10. Exact Best Subset for Small k

This is the direct version of:

```text
k=1: best single metric
k=2: best pair
k=3: best triple
```

Exact enumeration stops when the combination count is too large. Larger k is handled by forward/beam search below.


In [ ]:
exact_rows = []
for k in range(1, min(MAX_EXACT_K, len(SAFE_METRICS)) + 1):
    n_combos = int(math.comb(len(SAFE_METRICS), k))
    if n_combos > MAX_EXACT_COMBOS:
        print(f'Skip exact k={k}: {n_combos:,} combinations exceeds MAX_EXACT_COMBOS={MAX_EXACT_COMBOS:,}')
        continue

    print(f'Exact k={k}: testing {n_combos:,} combinations on train')
    best_row = None
    best_metrics = None
    best_train_macro = -np.inf

    for combo in combinations(SAFE_METRICS, k):
        row, _, _ = evaluate_combo_train_validation(combo, label=None, kind='exact_best_subset')
        if row['train_macro_power'] > best_train_macro:
            best_train_macro = row['train_macro_power']
            best_row = row
            best_metrics = combo

    best_row['label'] = f'exact best k={k}'
    best_row['method'] = 'exact'
    exact_rows.append(best_row)
    print(f'  Best: {best_row["display_metrics"]}')
    print(f'  Train power={best_row["train_overall_power"]:.3f}; validation power={best_row["validation_overall_power"]:.3f}; validation FPR={best_row["validation_FPR"]:.3f}')

exact_best = pd.DataFrame(exact_rows)
exact_best.to_csv(OUT_DIR / 'exact_best_subset_small_k.csv', index=False, float_format='%.6f')
print('\nExact best subset summary:')
print_eval_table(exact_best)


## 11. Backward Elimination (Ablation)

Start from all safe metrics, then iteratively remove the metric whose removal causes the smallest power drop (or largest power gain). This is the correct ablation approach: it reveals which metrics are truly redundant and which are load-bearing.

Key insight: with the max-Z joint test, more metrics means a higher null threshold (multiple-comparison penalty). So removing redundant metrics can actually *increase* power. Backward elimination naturally discovers the sweet spot.

In [ ]:
remaining = list(SAFE_METRICS)
backward_rows = []

# Step 0: full safe set baseline
row0, _, _ = evaluate_combo_train_validation(remaining, label=f'backward k={len(remaining)} (full)', kind='backward')
row0['method'] = 'backward'
row0['removed_metric'] = ''
row0['removed_display'] = '(start)'
backward_rows.append(row0)

# Iteratively remove the least impactful metric
while len(remaining) > 1:
    best_row = None
    best_metric_to_remove = None
    best_train_macro = -np.inf

    for candidate in remaining:
        combo = [m for m in remaining if m != candidate]
        row, _, _ = evaluate_combo_train_validation(combo, label=None, kind='backward')
        if row['train_macro_power'] > best_train_macro:
            best_train_macro = row['train_macro_power']
            best_row = row
            best_metric_to_remove = candidate

    remaining.remove(best_metric_to_remove)
    k = len(remaining)
    best_row['label'] = f'backward k={k}'
    best_row['method'] = 'backward'
    best_row['removed_metric'] = best_metric_to_remove
    best_row['removed_display'] = METRIC_DISPLAY[best_metric_to_remove]
    backward_rows.append(best_row)

    if k <= 10 or k % 5 == 0:
        print(f'k={k:2d}: remove {METRIC_DISPLAY[best_metric_to_remove]:25s} -> train power={best_row["train_overall_power"]:.4f}, val power={best_row["validation_overall_power"]:.4f}')

backward_path = pd.DataFrame(backward_rows)
backward_path.to_csv(OUT_DIR / 'backward_elimination.csv', index=False, float_format='%.6f')

# Show the interesting part: where power peaks and the final descent
print('\n── Backward elimination path (selected steps) ──')
print(backward_path[['label', 'removed_display', 'k', 'train_overall_power', 'validation_overall_power', 'validation_macro_power', 'validation_FPR']]
      .to_string(index=False, float_format='{:.4f}'.format))

# Find the peak
peak_idx = backward_path['train_macro_power'].idxmax()
peak_row = backward_path.loc[peak_idx]
print(f'\nPeak train power at k={int(peak_row["k"])}: train={peak_row["train_overall_power"]:.4f}, val={peak_row["validation_overall_power"]:.4f}')

val_peak_idx = backward_path['validation_macro_power'].idxmax()
val_peak_row = backward_path.loc[val_peak_idx]
print(f'Peak validation power at k={int(val_peak_row["k"])}: train={val_peak_row["train_overall_power"]:.4f}, val={val_peak_row["validation_overall_power"]:.4f}, macro={val_peak_row["validation_macro_power"]:.4f}')
print(f'  metrics: {val_peak_row["display_metrics"]}')

## 12. Beam Sparse Search

Beam search keeps the top train-performing partial combinations at each k and expands only those. It approximates best subset search for larger k without testing all `2^p` subsets.


In [ ]:
beam_rows = []
beam = [tuple()]
seen = set()

for k in range(1, min(MAX_SPARSE_K, len(SAFE_METRICS)) + 1):
    candidates = set()
    for combo in beam:
        start = 0
        for metric in SAFE_METRICS:
            if metric not in combo:
                candidates.add(tuple(sorted(combo + (metric,))))

    scored = []
    print(f'Beam k={k}: evaluating {len(candidates):,} candidates')
    for combo in candidates:
        row, _, _ = evaluate_combo_train_validation(combo, label=None, kind='beam')
        scored.append(row)

    scored_df = pd.DataFrame(scored).sort_values('train_macro_power', ascending=False)
    beam = [tuple(row.split(',')) for row in scored_df['metrics'].head(BEAM_WIDTH)]

    best_row = scored_df.iloc[0].to_dict()
    best_row['label'] = f'beam best k={k}'
    best_row['method'] = 'beam'
    beam_rows.append(best_row)

    print(f'  Best: {best_row["display_metrics"]}')
    print(f'  Train power={best_row["train_overall_power"]:.3f}; validation power={best_row["validation_overall_power"]:.3f}; validation FPR={best_row["validation_FPR"]:.3f}')

beam_best = pd.DataFrame(beam_rows)
beam_best.to_csv(OUT_DIR / 'beam_sparse_selection.csv', index=False, float_format='%.6f')
print('\nBeam best by k:')
print_eval_table(beam_best)


## 13. MINE-Specific Check

Since all metrics are now evaluated on the same 9K case set, this is a fair comparison:

- MINE-only performance
- full-safe without MINE
- best sparse combinations with vs without at least one MINE metric

In [ ]:
mine_metrics = [m for m in SAFE_METRICS if METRIC_GROUPS[m] == 'mine']
non_mine_metrics = [m for m in SAFE_METRICS if METRIC_GROUPS[m] != 'mine']

mine_rows = []
if mine_metrics:
    row, _, _ = evaluate_combo_train_validation(mine_metrics, label='MINE only', kind='mine_check')
    mine_rows.append(row)

if non_mine_metrics:
    row, _, _ = evaluate_combo_train_validation(non_mine_metrics, label='all_safe - MINE', kind='mine_check')
    row['validation_power_drop_vs_full_safe'] = FULL_SAFE_POWER - row['validation_overall_power']
    mine_rows.append(row)

# Compare sparse candidates that include MINE vs those that do not.
sparse_candidates = pd.concat([exact_best, backward_path, beam_best], ignore_index=True, sort=False)
sparse_candidates['has_mine'] = sparse_candidates['metrics'].fillna('').apply(
    lambda s: any(METRIC_GROUPS.get(m) == 'mine' for m in s.split(',') if m)
)

for has_mine, label in [(True, 'best sparse with MINE'), (False, 'best sparse without MINE')]:
    sub = sparse_candidates[sparse_candidates['has_mine'] == has_mine]
    if len(sub):
        best = sub.sort_values(['validation_macro_power', 'validation_FPR'], ascending=[False, True]).iloc[0].to_dict()
        best['label'] = label
        best['kind'] = 'mine_check'
        mine_rows.append(best)

mine_check = pd.DataFrame(mine_rows)
mine_check.to_csv(OUT_DIR / 'mine_specific_check.csv', index=False, float_format='%.6f')
print('MINE-specific checks:')
print_eval_table(mine_check)

## 14. Final Sparse Recommendation

Selection rule:

1. Use validation FPR <= `VALIDATION_FPR_LIMIT` (0.08).
2. Prefer combinations with validation power within `NEAR_FULL_TOL` (0.02 = 2 pp) of the full-combo reference power. (Sparse subsets often exceed this reference due to lower max-Z thresholds.)
3. Among those, choose the smallest k.
4. Tie-breaker: higher validation power, then lower validation FPR.

In [ ]:
all_sparse = pd.concat([exact_best, backward_path, beam_best], ignore_index=True, sort=False)
all_sparse = all_sparse.drop_duplicates(subset=['metrics', 'method'], keep='first')
all_sparse['validation_power_gap_vs_full_combo'] = FULL_SAFE_MACRO_POWER - all_sparse['validation_macro_power']
all_sparse['validation_macro_gap_vs_full_combo'] = FULL_SAFE_MACRO_POWER - all_sparse['validation_macro_power']
all_sparse.to_csv(OUT_DIR / 'all_sparse_candidates.csv', index=False, float_format='%.6f')

eligible = all_sparse[all_sparse['validation_FPR'] <= VALIDATION_FPR_LIMIT].copy()
if eligible.empty:
    eligible = all_sparse.copy()
    print(f'Warning: no sparse combo has validation FPR <= {VALIDATION_FPR_LIMIT:.2f}; selecting from all sparse candidates.')

near_full = eligible[eligible['validation_power_gap_vs_full_combo'] <= NEAR_FULL_TOL].copy()
if near_full.empty:
    best_val = eligible['validation_macro_power'].max()
    near_full = eligible[eligible['validation_macro_power'] >= best_val - NEAR_FULL_TOL].copy()
    print('No sparse combo is within tolerance of full-combo power; selecting near the best sparse validation power instead.')

recommended = near_full.sort_values(['k', 'validation_macro_power', 'validation_FPR'], ascending=[True, False, True]).iloc[0]
recommended_metrics = recommended['metrics'].split(',')

print('Recommended sparse combo:')
print(f'  method = {recommended.get("method", "unknown")}')
print(f'  k = {int(recommended["k"])}')
print(f'  metrics = {recommended["display_metrics"]}')
print(f'  validation overall power = {recommended["validation_overall_power"]:.3f}')
print(f'  validation macro power   = {recommended["validation_macro_power"]:.3f}')
print(f'  validation FPR = {recommended["validation_FPR"]:.3f}')
print(f'  gap vs full-combo: overall={recommended["validation_power_gap_vs_full_combo"]:.3f}, macro={recommended["validation_macro_gap_vs_full_combo"]:.3f}')
print(f'  breakdown: MO={recommended["validation_MO"]:.3f}, VO={recommended["validation_VO"]:.3f}, MV={recommended["validation_MV"]:.3f}')

with open(OUT_DIR / 'recommendation.txt', 'w') as f:
    f.write('B_S3.4 sparse metric recommendation\n')
    f.write(f'full_combo_validation_power = {FULL_SAFE_POWER:.6f}\n')
    f.write(f'full_combo_validation_macro_power = {FULL_SAFE_MACRO_POWER:.6f}\n')
    f.write(f'method = {recommended.get("method", "unknown")}\n')
    f.write(f'k = {int(recommended["k"])}\n')
    f.write(f'metrics = {recommended["display_metrics"]}\n')
    f.write(f'validation_overall_power = {recommended["validation_overall_power"]:.6f}\n')
    f.write(f'validation_macro_power = {recommended["validation_macro_power"]:.6f}\n')
    f.write(f'validation_FPR = {recommended["validation_FPR"]:.6f}\n')
    f.write(f'validation_MO = {recommended["validation_MO"]:.6f}\n')
    f.write(f'validation_VO = {recommended["validation_VO"]:.6f}\n')
    f.write(f'validation_MV = {recommended["validation_MV"]:.6f}\n')

## 15. Driver Analysis for the Recommended Combo

Drivers are counted only among detected validation cases. This avoids assigning a driver to cases that were not actually detected.


In [ ]:
threshold = fit_threshold(train, recommended_metrics, alpha=ALPHA)
val_eval = evaluate_fixed_threshold(validation, recommended_metrics, threshold)
validation_result = validation.copy()
validation_result['T_recommended'] = val_eval['T']
validation_result['detected_recommended'] = val_eval['detected']
validation_result['driver_metric'] = validation_result[recommended_metrics].idxmax(axis=1)

for cat in ['mean_only', 'variance_only', 'mean+variance']:
    sub = validation_result[(validation_result['category'] == cat) & validation_result['detected_recommended']]
    print(f'\nDriver metrics among detected validation {cat} cases (n={len(sub):,}):')
    if len(sub) == 0:
        continue
    counts = sub['driver_metric'].value_counts()
    for metric, n in counts.items():
        print(f'  {METRIC_DISPLAY[metric]:35s}  {n:6d}  ({n / len(sub):.1%})')

validation_result[['case_id', 'category', 'x_distribution', 'family_id', 'snr', 'spread_pattern',
                   'T_recommended', 'detected_recommended', 'driver_metric']].to_csv(
    OUT_DIR / 'validation_recommended_case_results.csv', index=False
)


## 15.5. SNR Diagnostic: Where Does the Recommended Combo Fail?

Low SNR is where detection is hardest and where metric choice matters most. This section shows validation power by SNR for the recommended combo and compares it against alternative combos to see if a different metric set does better at low SNR.

In [ ]:
# ── Compare top combos by SNR ──
# Pull the best combos from each search method at key k values,
# plus the recommended combo and individual top metrics.

compare_combos = {}

# 1. Recommended
compare_combos[f'recommended (k={len(recommended_metrics)})'] = recommended_metrics

# 2. Backward elimination: best at each key k
for k_target in [2, 3, 5]:
    bk = backward_path[backward_path['k'] == k_target]
    if len(bk):
        row = bk.iloc[0]
        metrics = row['metrics'].split(',')
        label = f'backward k={k_target}'
        compare_combos[label] = metrics

# 3. Beam best at k=2, k=3
for k_target in [2, 3]:
    bk = beam_best[beam_best['k'] == k_target]
    if len(bk):
        row = bk.iloc[0]
        metrics = row['metrics'].split(',')
        label = f'beam k={k_target}'
        if metrics != compare_combos.get(f'backward k={k_target}'):
            compare_combos[label] = metrics

# 4. Top individual non-dcov metrics for reference
for metric_name, z_col in [('mic', 'z_mic'), ('ew_dist_ks', 'z_ew_dist_ks')]:
    if z_col in SAFE_METRICS:
        compare_combos[f'{metric_name} only'] = [z_col]

# 5. Full safe set as lower reference
compare_combos['all_safe'] = SAFE_METRICS

# ── Compute validation power by SNR for each combo ──
snr_levels = sorted(validation['snr'].dropna().unique())
signal_val = validation[validation['category'] != 'true_null'].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, categories, title in [
    (axes[0], ['mean_only', 'variance_only', 'mean+variance'], 'All signal (MO+VO+MV)'),
    (axes[1], ['variance_only'], 'Variance-only (hardest)'),
]:
    sub = signal_val[signal_val['category'].isin(categories)]

    for label, metrics in compare_combos.items():
        threshold = fit_threshold(train, metrics, alpha=ALPHA)
        T = sub[metrics].max(axis=1)
        detected = (T > threshold).fillna(False)

        rates = []
        for snr in snr_levels:
            snr_mask = sub['snr'] == snr
            n = snr_mask.sum()
            if n > 0:
                rates.append(float(detected[snr_mask].mean()))
            else:
                rates.append(np.nan)

        is_rec = 'recommended' in label
        style = '-o' if is_rec else ('--' if 'only' in label or 'all_safe' in label else '-s')
        lw = 2.5 if is_rec else 1.2
        ax.plot(range(len(snr_levels)), rates, style, label=label,
                markersize=5, lw=lw, alpha=0.85)

    ax.set_xticks(range(len(snr_levels)))
    ax.set_xticklabels([f'{s:.2g}' for s in snr_levels], rotation=45, fontsize=8)
    ax.set_xlabel('SNR')
    ax.set_ylabel('Validation Power')
    ax.set_title(title)
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=7, loc='lower right')

fig.suptitle('Validation Power by SNR: Top Combos from Search', fontsize=13)
plt.tight_layout()
fig.savefig(OUT_DIR / 'fig4_snr_diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()

# Print low-SNR detail table
print('\nValidation power at lowest SNR levels (all signal):')
print(f'{"combo":<25s}  {"k":>2s}', end='')
for snr in snr_levels[:4]:
    print(f'  SNR={snr:<6.2g}', end='')
print(f'  {"overall":>8s}')
print('─' * 85)

for label, metrics in compare_combos.items():
    threshold = fit_threshold(train, metrics, alpha=ALPHA)
    T = signal_val[metrics].max(axis=1)
    detected = (T > threshold).fillna(False)
    print(f'{label:<25s}  {len(metrics):2d}', end='')
    for snr in snr_levels[:4]:
        snr_mask = signal_val['snr'] == snr
        rate = float(detected[snr_mask].mean()) if snr_mask.sum() > 0 else float('nan')
        print(f'  {rate:>8.1%}', end='')
    print(f'  {float(detected.mean()):>8.1%}')

## 16. Ablation Visualizations

Four views of the metric selection results:

- **A. Backward Elimination Curve**: power vs number of metrics — shows fewer metrics = higher power
- **B. Group-Only**: each metric family used alone — distance (dcor) dominates
- **C. Leave-One-Group-Out**: power drop when removing each group — only distance matters
- **D. Leave-One-Metric-Out**: power drop when removing individual metrics — dcor is irreplaceable

In [ ]:
POWER_LABEL = 'Validation Power' if USE_SPLIT else 'Power'

from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

GROUP_LABELS = {
    'correlation': 'Correlation',
    'distance': 'Distance (dcor)',
    'slope': 'Slope',
    'bin': 'Binning',
    'distribution': 'Distribution',
    'mine': 'MINE',
    'nonlinear': 'Nonlinear',
}

# ═══════════════════════════════════════════════════
# Figure A: Backward Elimination Curve
# ═══════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 5.5))

bk = backward_path.copy()
ks = bk['k'].values
powers = bk['validation_overall_power'].values
k_max = int(ks[0])

bk['power_change'] = bk['validation_overall_power'].diff()
bk.loc[bk.index[0], 'power_change'] = 0

ax.fill_between(ks, powers, alpha=0.08, color='#2563eb')
ax.plot(ks, powers, '-o', color='#2563eb', markersize=5, lw=2.2, zorder=4,
        markerfacecolor='white', markeredgewidth=1.5)

ax.axhline(FULL_SAFE_POWER, color='#94a3b8', ls='--', lw=1, alpha=0.6)
ax.text(k_max - 0.3, FULL_SAFE_POWER + 0.0002, f'All {k_max} metrics: {FULL_SAFE_POWER:.4f}',
        fontsize=8.5, color='#64748b', ha='left', va='bottom')

peak_idx = bk['validation_macro_power'].idxmax()
peak_row = bk.loc[peak_idx]
peak_k = int(peak_row['k'])
peak_power = peak_row['validation_overall_power']
peak_metrics = peak_row['display_metrics']

ax.plot(peak_k, peak_power, '*', color='#2563eb', markersize=14, zorder=6)

# k=4 drop
k4_row = bk[bk['k'] == 4].iloc[0]
k4_power = k4_row['validation_overall_power']
k4_removed = k4_row['removed_display']
k4_change = k4_row['power_change']
ax.plot(4, k4_power, 'o', color='#ea580c', markersize=7, zorder=6, markeredgecolor='white', markeredgewidth=1.5)
ax.annotate(
    f'k=4: −{k4_removed}\npower {k4_change:+.2%}',
    xy=(4, k4_power),
    xytext=(7, k4_power - 0.0025),
    fontsize=9, color='#ea580c', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#ea580c', lw=1.5,
                    connectionstyle='arc3,rad=-0.2'),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff7ed',
              edgecolor='#ea580c', alpha=0.95),
    zorder=7)

# peak annotation — next to star
ax.annotate(
    f'Peak: k={peak_k}, power={peak_power:.4f}\nmetric: {peak_metrics}',
    xy=(peak_k, peak_power),
    xytext=(peak_k + 3.5, peak_power + 0.0002),
    fontsize=9, color='#1d4ed8', fontweight='bold',
    arrowprops=dict(arrowstyle='->', color='#1d4ed8', lw=1.5),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#eff6ff',
              edgecolor='#1d4ed8', alpha=0.95),
    zorder=7)

# step labels — horizontal, above each point
for _, row in bk.iterrows():
    k = int(row['k'])
    removed = row['removed_display']
    if removed == '(start)' or k == peak_k or k == 4:
        continue
    ax.text(k, row['validation_overall_power'] + 0.0003, f'−{removed}',
            fontsize=9, color='#64748b', ha='center', va='bottom')

ax.set_xlabel('Number of metrics (k)', fontsize=11, labelpad=8)
ax.set_ylabel(POWER_LABEL, fontsize=11, labelpad=8)
ax.set_title('Backward Elimination: Power at Each Removal Step', fontsize=13, fontweight='bold', pad=12)
ax.set_xlim(k_max + 0.5, 0.5)
y_range = powers.max() - powers.min()
ax.set_ylim(powers.min() - y_range * 0.12, powers.max() + y_range * 0.15)
ax.grid(True, alpha=0.15, ls='-')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=10)

plt.tight_layout()
fig.savefig(OUT_DIR / 'ablation_fig1_backward.png', dpi=200, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 6.5))

bk = backward_path.copy()
ks = bk['k'].values
powers = bk['validation_overall_power'].values
k_max = int(ks[0])

ax.plot(ks, powers, '-o', color='#377eb8', markersize=4, lw=2, zorder=3)
ax.axhline(FULL_SAFE_POWER, color='gray', ls=':', lw=1, alpha=0.7)

bk['power_change'] = bk['validation_overall_power'].diff()
bk.loc[bk.index[0], 'power_change'] = 0

peak_idx = bk['validation_macro_power'].idxmax()
peak_row = bk.loc[peak_idx]
peak_k = int(peak_row['k'])
peak_power = peak_row['validation_overall_power']
peak_metrics = peak_row['display_metrics']

k1_row = bk[bk['k'] == 1].iloc[0]
k1_power = k1_row['validation_overall_power']
k1_metrics = k1_row['display_metrics']
k1_removed = k1_row['removed_display']

notable = bk[(bk['power_change'].abs() > 0.0015) & (bk['removed_display'] != '(start)')]

for _, row in notable.iterrows():
    k = int(row['k'])
    if k == 1:
        continue
    removed = row['removed_display']
    power = row['validation_overall_power']
    change = row['power_change']
    sign = '↑' if change > 0 else '↓'
    text = f'{sign} −{removed}\n   {power:.3f} ({change:+.1%})'
    ax.annotate(text, xy=(k, power), xytext=(k + 3, power - 0.004),
                fontsize=8, color='#22c55e', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#22c55e', lw=1),
                bbox=dict(boxstyle='round,pad=0.2', facecolor='#f0fdf4', edgecolor='#22c55e', alpha=0.9),
                zorder=5)

ax.annotate(f'★ Peak: k={peak_k}, power={peak_power:.3f}\n   metrics: {peak_metrics}',
            xy=(peak_k, peak_power), xytext=(15, peak_power + 0.002),
            fontsize=9.5, color='#1d4ed8', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#1d4ed8', lw=2),
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#eff6ff', edgecolor='#1d4ed8', alpha=0.95),
            zorder=6)

ax.annotate(f'k=1: power={k1_power:.3f}  ({k1_metrics})\n'
            f'   ↓ −{k1_removed} drops {k1_power - peak_power:+.1%}',
            xy=(1, k1_power), xytext=(12, k1_power - 0.005),
            fontsize=9.5, color='#dc2626', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#dc2626', lw=2),
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#fef2f2', edgecolor='#dc2626', alpha=0.95),
            zorder=6)

ax.text(k_max - 2, FULL_SAFE_POWER - 0.001, f'All {k_max} metrics\n{FULL_SAFE_POWER:.3f}',
        fontsize=9, color='gray', ha='center')

ax.set_xlabel('Number of metrics (k)', fontsize=12)
ax.set_ylabel(POWER_LABEL, fontsize=12)
ax.set_title('Backward Elimination: Power Change at Each Removal Step', fontsize=14, fontweight='bold')
ax.set_xlim(k_max + 1, 1)
y_min = min(FULL_SAFE_POWER, k1_power) - 0.008
y_max = peak_power + 0.006
ax.set_ylim(y_min, y_max)
ax.grid(True, alpha=0.2)

plt.tight_layout()
fig.savefig(OUT_DIR / 'ablation_fig1_backward.png', dpi=200, bbox_inches='tight')
plt.show()

# Figure B: Group-Only Performance
# ═══════════════════════════════════════════════════
group_only = group_ablation[group_ablation['kind'] == 'group_only'].copy()
group_only['group'] = group_only['label'].str.replace(' only', '')
group_only = group_only.sort_values('validation_overall_power', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
groups_b = group_only['group'].values
powers_b = group_only['validation_overall_power'].values * 100
colors_b = [GROUP_COLORS.get(g, '#999') for g in groups_b]
labels_b = [GROUP_LABELS.get(g, g) for g in groups_b]

ax.barh(range(len(groups_b)), powers_b, color=colors_b, alpha=0.85, height=0.6, edgecolor='white', lw=0.5)
for i, p in enumerate(powers_b):
    ax.text(p + 0.5, i, f'{p:.1f}%', va='center', fontsize=10, fontweight='bold')

ax.set_yticks(range(len(groups_b)))
ax.set_yticklabels(labels_b, fontsize=11)
ax.axvline(FULL_SAFE_POWER * 100, color='gray', ls=':', lw=1.2)
ax.text(FULL_SAFE_POWER * 100 + 0.3, len(groups_b) - 0.5, f'All combined\n{FULL_SAFE_POWER*100:.1f}%',
        fontsize=9, color='gray', va='top')
ax.set_xlabel(f'{POWER_LABEL} (%)', fontsize=12)
ax.set_xlim(0, 105)
ax.set_title('Each Metric Group Used Alone', fontsize=14, fontweight='bold')
ax.grid(True, axis='x', alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(OUT_DIR / 'ablation_fig2_group_only.png', dpi=200, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════
# Figure C: Leave-One-Group-Out
# ═══════════════════════════════════════════════════
leave_group_plot = leave_group.copy()
leave_group_plot['group'] = leave_group_plot['label'].str.replace('all_safe - ', '')
leave_group_plot = leave_group_plot.sort_values('validation_power_drop_vs_full_safe', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
groups_c = leave_group_plot['group'].values
drops_c = leave_group_plot['validation_power_drop_vs_full_safe'].values * 100
colors_c = [GROUP_COLORS.get(g, '#999') for g in groups_c]
labels_c = [GROUP_LABELS.get(g, g) for g in groups_c]

ax.barh(range(len(groups_c)), drops_c, color=colors_c, alpha=0.85, height=0.6, edgecolor='white', lw=0.5)
for i, (d, g) in enumerate(zip(drops_c, groups_c)):
    if d > 0:
        ax.text(d + 0.05, i, f'+{d:.1f}%', va='center', fontsize=10, fontweight='bold')
    else:
        ax.text(d - 0.05, i, f'{d:.1f}%', va='center', fontsize=10, ha='right')

ax.set_yticks(range(len(groups_c)))
ax.set_yticklabels(labels_c, fontsize=11)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Power drop (percentage points)', fontsize=12)
ax.set_title('Leave-One-Group-Out: Which Metric Group Matters?', fontsize=14, fontweight='bold')
ax.grid(True, axis='x', alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(OUT_DIR / 'ablation_fig3_group_drop.png', dpi=200, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════
# Figure D: Leave-One-Metric-Out (top 15)
# ═══════════════════════════════════════════════════
top_n = 15
top_metrics = metric_ablation.head(top_n).copy()
top_metrics = top_metrics.sort_values('validation_power_drop_vs_full_safe', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
drops_d = top_metrics['validation_power_drop_vs_full_safe'].values * 100
names_d = top_metrics['display_removed'].values
groups_d = top_metrics['group_removed'].values
colors_d = [GROUP_COLORS.get(g, '#999') for g in groups_d]

ax.barh(range(len(names_d)), drops_d, color=colors_d, alpha=0.85, height=0.6, edgecolor='white', lw=0.5)
for i, d in enumerate(drops_d):
    if d > 0.05:
        ax.text(d + 0.03, i, f'+{d:.2f}%', va='center', fontsize=9, fontweight='bold')

ax.set_yticks(range(len(names_d)))
ax.set_yticklabels(names_d, fontsize=10)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Power drop (percentage points)', fontsize=12)
ax.set_title('Leave-One-Metric-Out: Top 15 Individual Contributions', fontsize=14, fontweight='bold')
ax.grid(True, axis='x', alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

seen_groups = []
legend_handles = []
for g in groups_d[::-1]:
    if g not in seen_groups:
        seen_groups.append(g)
        legend_handles.append(Patch(facecolor=GROUP_COLORS.get(g, '#999'), label=GROUP_LABELS.get(g, g), alpha=0.85))
ax.legend(handles=legend_handles[::-1], fontsize=9, loc='lower right', title='Group', title_fontsize=10)

plt.tight_layout()
fig.savefig(OUT_DIR / 'ablation_fig4_metric_drop.png', dpi=200, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════
# Figure E: Combined Summary (A+B+C)
# ═══════════════════════════════════════════════════
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)

# Panel A: Backward elimination (validation only)
ax = fig.add_subplot(gs[0, :])
ax.plot(bk['k'], bk['validation_overall_power'], '-o',
        color='#377eb8', markersize=4, lw=2, zorder=3)
ax.axhline(FULL_SAFE_POWER, color='gray', ls=':', lw=1, alpha=0.7)
ax.text(33, FULL_SAFE_POWER + 0.0015, f'All 35: {FULL_SAFE_POWER:.3f}', fontsize=8, color='gray')
peak_idx = bk['validation_macro_power'].idxmax()
pk = bk.loc[peak_idx, 'k']
pp = bk.loc[peak_idx, 'validation_overall_power']
ax.annotate(f'k={int(pk)}: {pp:.3f}', xy=(pk, pp), xytext=(6, 0.968),
            fontsize=10, fontweight='bold', color='#377eb8',
            arrowprops=dict(arrowstyle='->', color='#377eb8', lw=1.5))
ax.set_xlabel('Number of metrics (k)')
ax.set_ylabel(POWER_LABEL)
ax.set_title('A. Backward Elimination: Fewer Metrics → Higher Power', fontsize=13, fontweight='bold')
ax.set_xlim(36, 1)
ax.set_ylim(0.955, 0.99)
ax.grid(True, alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel B: Group-only
ax = fig.add_subplot(gs[1, 0])
ax.barh(range(len(groups_b)), powers_b, color=colors_b, alpha=0.85, height=0.6)
for i, p in enumerate(powers_b):
    ax.text(p + 0.5, i, f'{p:.1f}%', va='center', fontsize=9)
ax.set_yticks(range(len(groups_b)))
ax.set_yticklabels(labels_b, fontsize=10)
ax.axvline(FULL_SAFE_POWER * 100, color='gray', ls=':', lw=1)
ax.set_xlabel(f'{POWER_LABEL} (%)')
ax.set_xlim(0, 105)
ax.set_title('B. Each Group Alone', fontsize=13, fontweight='bold')
ax.grid(True, axis='x', alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Panel C: Leave-one-group-out
ax = fig.add_subplot(gs[1, 1])
ax.barh(range(len(groups_c)), drops_c, color=colors_c, alpha=0.85, height=0.6)
for i, (d, g) in enumerate(zip(drops_c, groups_c)):
    if d > 0:
        ax.text(d + 0.05, i, f'+{d:.1f}%', va='center', fontsize=9, fontweight='bold')
    else:
        ax.text(d - 0.05, i, f'{d:.1f}%', va='center', fontsize=9, ha='right')
ax.set_yticks(range(len(groups_c)))
ax.set_yticklabels(labels_c, fontsize=10)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Power drop (pp)')
ax.set_title('C. Remove One Group: Power Drop', fontsize=13, fontweight='bold')
ax.grid(True, axis='x', alpha=0.2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.suptitle('Metric Selection: Ablation Experiments', fontsize=16, fontweight='bold', y=0.98)
fig.savefig(OUT_DIR / 'ablation_fig5_summary.png', dpi=200, bbox_inches='tight')
plt.show()

## 16.5 FPR Verification

For each k, the best metric combo's threshold is calibrated from true-null cases.
If FPR stays flat near α = 0.05 while power varies, the comparison is fair.

In [ ]:
# ═══════════════════════════════════════════════════
# Best combo at each k: Power vs FPR
# ═══════════════════════════════════════════════════

# Collect best combo per k across all search methods
all_by_k = pd.concat([exact_best, backward_path, beam_best], ignore_index=True, sort=False)
best_per_k = all_by_k.sort_values('validation_macro_power', ascending=False).drop_duplicates('k', keep='first')
best_per_k = best_per_k.sort_values('k')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1], 'hspace': 0.08})

ks_plot = best_per_k['k'].values
powers_plot = best_per_k['validation_overall_power'].values
fprs_plot = best_per_k['validation_FPR'].values

# Top: Power
ax1.plot(ks_plot, powers_plot, '-o', color='#2563eb', markersize=6, lw=2.2, zorder=4,
         markerfacecolor='white', markeredgewidth=1.5)
ax1.axhline(FULL_SAFE_POWER, color='#94a3b8', ls='--', lw=1, alpha=0.6)
ax1.text(ks_plot[-1], FULL_SAFE_POWER + 0.0005,
         f'All {len(SAFE_METRICS)} metrics: {FULL_SAFE_POWER:.4f}',
         fontsize=9, color='#64748b', ha='right', va='bottom')

peak_idx = best_per_k['validation_macro_power'].idxmax()
pk = best_per_k.loc[peak_idx]
ax1.plot(pk['k'], pk['validation_overall_power'], '*', color='#2563eb', markersize=14, zorder=6)
ax1.annotate(f'Peak k={int(pk["k"])}: {pk["validation_overall_power"]:.4f}',
             xy=(pk['k'], pk['validation_overall_power']),
             xytext=(pk['k'] + 2, pk['validation_overall_power'] + 0.001),
             fontsize=9, color='#1d4ed8', fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='#1d4ed8', lw=1.5),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#eff6ff',
                       edgecolor='#1d4ed8', alpha=0.95), zorder=7)

for _, row in best_per_k.iterrows():
    k = int(row['k'])
    if k == int(pk['k']):
        continue
    ax1.text(k, row['validation_overall_power'] + 0.0005,
             f'{row["validation_overall_power"]:.4f}',
             fontsize=7.5, color='#64748b', ha='center', va='bottom')

ax1.set_ylabel(POWER_LABEL, fontsize=11, labelpad=8)
ax1.set_title('Best Metric Combo at Each k: Power & FPR',
              fontsize=13, fontweight='bold', pad=12)
ax1.grid(True, alpha=0.15)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.tick_params(labelsize=10)

# Bottom: FPR
ax2.plot(ks_plot, fprs_plot, '-s', color='#dc2626', markersize=6, lw=2.2, zorder=4,
         markerfacecolor='white', markeredgewidth=1.5)
ax2.axhline(ALPHA, color='#64748b', ls='--', lw=1.5, alpha=0.8)
ax2.text(ks_plot[-1], ALPHA + 0.002, f'α = {ALPHA}',
         fontsize=10, color='#64748b', ha='right', va='bottom', fontweight='bold')

for _, row in best_per_k.iterrows():
    ax2.text(int(row['k']), row['validation_FPR'] + 0.002,
             f'{row["validation_FPR"]:.3f}',
             fontsize=8, color='#991b1b', ha='center', va='bottom')

ax2.set_xlabel('Number of metrics (k)', fontsize=11, labelpad=8)
ax2.set_ylabel('FPR', fontsize=11, labelpad=8)
ax2.set_xlim(0.5, ks_plot[-1] + 0.5)
fpr_range = max(fprs_plot.max() - fprs_plot.min(), 0.02)
ax2.set_ylim(max(0, fprs_plot.min() - fpr_range * 0.5),
             fprs_plot.max() + fpr_range * 0.5)
ax2.grid(True, alpha=0.15)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.tick_params(labelsize=10)

plt.tight_layout()
fig.savefig(OUT_DIR / 'best_per_k_power_fpr.png', dpi=200, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════
# Backward elimination: Power + FPR dual panel
# ═══════════════════════════════════════════════════
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1], 'hspace': 0.08})

bk_fpr = backward_path.copy()
ks_bk = bk_fpr['k'].values
powers_bk = bk_fpr['validation_overall_power'].values
fprs_bk = bk_fpr['validation_FPR'].values

# Top: Power
ax1.fill_between(ks_bk, powers_bk, alpha=0.08, color='#2563eb')
ax1.plot(ks_bk, powers_bk, '-o', color='#2563eb', markersize=5, lw=2.2, zorder=4,
         markerfacecolor='white', markeredgewidth=1.5)
ax1.axhline(FULL_SAFE_POWER, color='#94a3b8', ls='--', lw=1, alpha=0.6)
ax1.set_ylabel(POWER_LABEL, fontsize=11, color='#2563eb', labelpad=8)
ax1.set_title('Backward Elimination: Power & FPR at Each Step',
              fontsize=13, fontweight='bold', pad=12)
ax1.grid(True, alpha=0.15)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.tick_params(labelsize=10)

# Bottom: FPR
ax2.fill_between(ks_bk, fprs_bk, alpha=0.08, color='#dc2626')
ax2.plot(ks_bk, fprs_bk, '-s', color='#dc2626', markersize=5, lw=2.2, zorder=4,
         markerfacecolor='white', markeredgewidth=1.5)
ax2.axhline(ALPHA, color='#64748b', ls='--', lw=1.5, alpha=0.8)
ax2.text(ks_bk[0], ALPHA + 0.002, f'α = {ALPHA}',
         fontsize=10, color='#64748b', ha='left', va='bottom', fontweight='bold')
ax2.set_xlabel('Number of metrics (k)', fontsize=11, labelpad=8)
ax2.set_ylabel('FPR', fontsize=11, color='#dc2626', labelpad=8)
ax2.set_xlim(int(ks_bk[0]) + 0.5, 0.5)
fpr_range_bk = max(fprs_bk.max() - fprs_bk.min(), 0.02)
ax2.set_ylim(max(0, fprs_bk.min() - fpr_range_bk * 0.5),
             fprs_bk.max() + fpr_range_bk * 0.5)
ax2.grid(True, alpha=0.15)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.tick_params(labelsize=10)

plt.tight_layout()
fig.savefig(OUT_DIR / 'backward_power_fpr_dual.png', dpi=200, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════
# Group-Only: Power vs FPR side by side
# ═══════════════════════════════════════════════════
group_only_plot = group_ablation[group_ablation['kind'] == 'group_only'].copy()
group_only_plot['group'] = group_only_plot['label'].str.replace(' only', '')
group_only_plot = group_only_plot.sort_values('validation_overall_power', ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

groups_g = group_only_plot['group'].values
powers_g = group_only_plot['validation_overall_power'].values * 100
fprs_g = group_only_plot['validation_FPR'].values * 100
colors_g = [GROUP_COLORS.get(g, '#999') for g in groups_g]
labels_g = [GROUP_LABELS.get(g, g) for g in groups_g]

ax1.barh(range(len(groups_g)), powers_g, color=colors_g, alpha=0.85, height=0.6)
for i, p in enumerate(powers_g):
    ax1.text(p + 0.5, i, f'{p:.1f}%', va='center', fontsize=9, fontweight='bold')
ax1.set_yticks(range(len(groups_g)))
ax1.set_yticklabels(labels_g, fontsize=10)
ax1.axvline(FULL_SAFE_POWER * 100, color='gray', ls=':', lw=1.2)
ax1.set_xlabel(f'{POWER_LABEL} (%)', fontsize=11)
ax1.set_xlim(0, 105)
ax1.set_title('Power (each group alone)', fontsize=12, fontweight='bold')
ax1.grid(True, axis='x', alpha=0.2)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2.barh(range(len(groups_g)), fprs_g, color=colors_g, alpha=0.85, height=0.6)
for i, fpr in enumerate(fprs_g):
    ax2.text(fpr + 0.2, i, f'{fpr:.1f}%', va='center', fontsize=9, fontweight='bold')
ax2.axvline(ALPHA * 100, color='#dc2626', ls='--', lw=1.5)
ax2.text(ALPHA * 100 + 0.3, len(groups_g) - 0.3, f'α = {ALPHA*100:.0f}%',
         fontsize=9, color='#dc2626', va='top', fontweight='bold')
ax2.set_xlabel('FPR (%)', fontsize=11)
ax2.set_xlim(0, max(fprs_g.max() * 1.5, ALPHA * 100 * 2))
ax2.set_title('FPR (each group alone)', fontsize=12, fontweight='bold')
ax2.grid(True, axis='x', alpha=0.2)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle('Group-Only: Power vs FPR', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(OUT_DIR / 'group_only_power_fpr.png', dpi=200, bbox_inches='tight')
plt.show()

## 17. Summary


## Reading the Output Columns

Some output column names still say `FPR`, for example `validation_FPR`. In this notebook, read them as:

```text
fraction of true_null cases with p <= alpha
```

That is the empirical false-positive rate induced by the p-value cutoff. It is reported to verify that the selected rule is calibrated, not because it is a separate algorithm.


In [ ]:
print('=' * 90)
print('B_S3.4 Metric Selection Summary')
print('=' * 90)
print(f'Cases: {len(df):,} (MINE-covered subset of B_S3 17K output)')
print(f'Train: {len(train):,}; Validation: {len(validation):,}')
print(f'Safe metrics: {len(SAFE_METRICS)} / {len(Z_COLS)} (all full-coverage)')
print(f'Full-combo reference: overall={FULL_SAFE_POWER:.3f}, macro={FULL_SAFE_MACRO_POWER:.3f}')
print()
print('Note: overall_power is weighted by case count (VO dominates at ~60%).')
print('      macro_power = mean(MO, VO, MV) gives equal weight to each relationship type.')
print()
print('Full-combo references:')
print_eval_table(benchmarks)
print()
print('Most important groups by leave-one-group-out drop:')
print(leave_group[['group_removed', 'validation_power_drop_vs_full_safe', 'validation_overall_power', 'validation_macro_power', 'validation_FPR']]
      .to_string(index=False, float_format='{:.4f}'.format))
print()
print('Most important metrics by leave-one-metric-out drop:')
print(metric_ablation[['display_removed', 'group_removed', 'validation_power_drop_vs_full_safe']]
      .head(15).to_string(index=False, float_format='{:.4f}'.format))
print()
print('── Backward elimination: key steps ──')
max_k = int(backward_path['k'].max())
backward_key = backward_path[backward_path['k'].isin([max_k, 30, 20, 15, 10, 8, 6, 5, 4, 3, 2, 1])]
print(backward_key[['label', 'k', 'display_metrics', 'validation_overall_power', 'validation_macro_power', 'validation_FPR']]
      .to_string(index=False, float_format='{:.4f}'.format))
print()
print('Recommended sparse combo:')
print(f'  method: {recommended.get("method", "unknown")}')
print(f'  k: {int(recommended["k"])}')
print(f'  metrics: {recommended["display_metrics"]}')
print(f'  validation overall power: {recommended["validation_overall_power"]:.3f}')
print(f'  validation macro power:   {recommended["validation_macro_power"]:.3f}')
print(f'  validation FPR: {recommended["validation_FPR"]:.3f}')
print(f'  gap vs full-combo: overall={recommended["validation_power_gap_vs_full_combo"]:.3f}, macro={recommended["validation_macro_gap_vs_full_combo"]:.3f}')
print(f'  breakdown: MO={recommended["validation_MO"]:.3f}, VO={recommended["validation_VO"]:.3f}, MV={recommended["validation_MV"]:.3f}')
print(f'\nSaved outputs to {OUT_DIR}')